# CV parser: InternVL3-1B LoRA full run
Runtime → Change runtime type → **A100** (or L4). T4 has no bf16; the script stops on it.
Recipe: `FINETUNING_RECIPE.md` in the project. Output goes to `MyDrive/cv_parser_colab/runs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy data to the local disk (Drive reads are slow), same layout as the project
!cp /content/drive/MyDrive/cv_parser_colab/cv_parser_colab.zip /content/ && cd /content && unzip -q -o cv_parser_colab.zip && ls /content/cv_parser/data/processed

In [ ]:
# Same versions as the Mac where the script was tested
!pip -q install transformers==4.57.6 peft==0.21.2
import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__)

In [ ]:
# Short check first (~5 min): 20 pages, 1 epoch, no generation. Must print 'mask check ok' and falling losses.
%cd /content/cv_parser
!python src/train_lora.py --limit 20 --epochs 1 --accum 4 --check-pages 0 --out /content/smoke

In [ ]:
# Full run: all training pages, 2 epochs, effective batch 16, then 10-page generation check
%cd /content/cv_parser
!python src/train_lora.py --out /content/drive/MyDrive/cv_parser_colab/runs/lora_v1 2>&1 | tee /content/drive/MyDrive/cv_parser_colab/runs_lora_v1.log